# Beginner Databricks Flow: Census Data Preprocessing to Delta

This notebook follows a simple Databricks learning flow:

1. Import the dataset
2. Understand the data
3. Preprocess the data using normal ML/data preparation techniques
4. Save the cleaned data in a final DataFrame
5. Save the final DataFrame as a Delta table

## Step 1: Set Input and Output Paths

Upload `census_data_raw.csv` to Databricks first. A common upload location is `dbfs:/FileStore/tables/census_data_raw.csv`.

In [ ]:
dbutils.widgets.text("raw_path", "dbfs:/FileStore/tables/census_data_raw.csv")
dbutils.widgets.text("target_table", "workshop.default.donation_data_v1")

raw_path = dbutils.widgets.get("raw_path")
target_table = dbutils.widgets.get("target_table")

print("Input file:", raw_path)
print("Output Delta table:", target_table)


## Step 2: Import the Dataset

Read the CSV file with headers. At the start, keep columns as strings because the raw file contains dirty values like `$`, `%`, `k`, blanks, and mixed date formats.

In [ ]:
raw_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(raw_path)
)

display(raw_df.limit(20))


## Step 3: Understand the Dataset

Check row count, column names, schema, and simple null counts before cleaning.

In [ ]:
from pyspark.sql import functions as F
import re

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))
raw_df.printSchema()


In [ ]:
null_counts_df = raw_df.select([
    F.sum(
        F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)
    ).alias(c)
    for c in raw_df.columns
])

display(null_counts_df)


## Step 4: Standardize Column Names

Make column names lowercase, remove spaces/special characters, and keep a fixed column order.

In [ ]:
def standardize_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

standard_columns = [standardize_column_name(c) for c in raw_df.columns]
df = raw_df.toDF(*standard_columns)

expected_columns = [
    "age", "workclass", "education_level", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income", "event_time", "random_flag",
    "source_system"
]

missing_columns = sorted(set(expected_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

df = df.select(*expected_columns)
display(df.limit(20))


## Step 5: Clean and Preprocess the Data

This is the main preprocessing step. It uses common ML preparation ideas: clean types, handle missing values, normalize labels, parse dates, and remove duplicates.

In [ ]:
def trim_to_null(column_name):
    value = F.trim(F.col(column_name).cast("string"))
    return F.when(value.isNull() | (value == "") | F.lower(value).isin("null", "none", "nan"), None).otherwise(value)

def clean_age(column_name):
    value = F.lower(F.trim(F.col(column_name).cast("string")))
    digits = F.regexp_extract(value, "([0-9]+)", 1)
    return F.when(value.isNull() | (value == "") | value.rlike("^error_?") | (digits == ""), None).otherwise(digits.cast("int"))

def clean_numeric(column_name):
    value = F.lower(F.trim(F.col(column_name).cast("string")))
    without_commas = F.regexp_replace(value, ",", "")
    multiplier = F.when(without_commas.rlike("k$"), F.lit(1000.0)).otherwise(F.lit(1.0))
    numeric_text = F.regexp_replace(without_commas, "[^0-9.-]", "")
    is_valid_number = numeric_text.rlike("^-?[0-9]+(\\.[0-9]+)?$")
    return F.when(value.isNull() | (value == "") | value.rlike("^error_?") | ~is_valid_number, None).otherwise(numeric_text.cast("double") * multiplier)

def normalize_income(column_name):
    value = F.lower(F.regexp_replace(F.trim(F.col(column_name).cast("string")), "\\s+", ""))
    return (
        F.when(value.isin("<=80k", "=<80k", "le80k", "lessorequal80k", "low", "0"), "<=80k")
        .when(value.isin(">80k", "gt80k", "greaterthan80k", "high", "1"), ">80k")
        .otherwise(None)
    )


In [ ]:
clean_df = df

for column_name in clean_df.columns:
    clean_df = clean_df.withColumn(column_name, trim_to_null(column_name))

event_text = F.trim(F.col("event_time").cast("string"))

clean_df = (
    clean_df
    .withColumn("age", clean_age("age"))
    .withColumn("workclass", F.coalesce(F.col("workclass"), F.lit("Unknown")))
    .withColumn("education_num", clean_numeric("education_num"))
    .withColumn("capital_gain", clean_numeric("capital_gain"))
    .withColumn("capital_loss", clean_numeric("capital_loss"))
    .withColumn("hours_per_week", clean_numeric("hours_per_week"))
    .withColumn("income", normalize_income("income"))
    .withColumn(
        "event_time_std",
        F.coalesce(
            F.expr("try_to_timestamp(event_time)"),
            F.when(event_text.rlike("^[0-9]{10}$"), F.to_timestamp(F.from_unixtime(event_text.cast("bigint")))),
            F.when(event_text.rlike("^[0-9]{13}$"), F.to_timestamp(F.from_unixtime((event_text.cast("double") / F.lit(1000)).cast("bigint")))),
            F.expr("try_to_timestamp(event_time, 'dd/MM/yy HH:mm')"),
            F.expr("try_to_timestamp(event_time, 'dd/MM/yy')"),
            F.expr("try_to_timestamp(event_time, 'MM/dd/yy HH:mm')"),
            F.expr("try_to_timestamp(event_time, 'MM/dd/yy')")
        )
    )
)

display(clean_df.limit(20))


## Step 6: Remove Duplicate Records

Use business columns to remove duplicate rows after cleaning.

In [ ]:
business_key_columns = [
    "age", "workclass", "education_level", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "native_country", "event_time_std",
    "income", "source_system"
]

before_dedup_count = clean_df.count()
clean_df = clean_df.dropDuplicates(business_key_columns)
after_dedup_count = clean_df.count()

print("Rows before duplicate removal:", before_dedup_count)
print("Rows after duplicate removal:", after_dedup_count)
print("Duplicate rows removed:", before_dedup_count - after_dedup_count)


## Step 7: Create the Final DataFrame

Select only the final cleaned columns. This DataFrame is ready for ML, analytics, dashboards, or API use.

In [ ]:
final_df = (
    clean_df.select(
        "age", "workclass", "education_level", "education_num", "marital_status",
        "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
        "hours_per_week", "native_country", "income", "event_time_std", "random_flag",
        "source_system"
    )
    .withColumn("processed_at", F.current_timestamp())
)

final_df.printSchema()
display(final_df.limit(20))


## Step 8: Basic Data Quality Checks

Before saving, check nulls, labels, date parse failures, and row counts.

In [ ]:
final_null_counts_df = final_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in final_df.columns
])

display(final_null_counts_df)
display(final_df.groupBy("income").count())

timestamp_parse_failures = final_df.filter(F.col("event_time_std").isNull()).count()
print("Final row count:", final_df.count())
print("Timestamp parse failures:", timestamp_parse_failures)


## Step 9: Save the Final DataFrame to Delta

This writes `final_df` into the required Delta table using overwrite mode, so the notebook can be rerun safely.

In [ ]:
parts = target_table.split(".")
if len(parts) == 3:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {parts[0]}.{parts[1]}")
elif len(parts) == 2:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {parts[0]}")

(
    final_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Saved final DataFrame as Delta table:", target_table)


## Step 10: Validate the Delta Table

Read the Delta table back and compare its row count with `final_df`.

In [ ]:
delta_df = spark.table(target_table)

if delta_df.count() != final_df.count():
    raise ValueError("Delta table row count does not match final_df row count")

display(delta_df.limit(20))
print("Delta table validation passed")
